# Cognee + Google ADK Integration Guide

This notebook demonstrates how to integrate **Cognee** (a semantic memory for AI agents) with **Google Agent Development Kit (ADK)** to create AI agents with persistent, cross-session memory capabilities powered by knowledge graphs + embeddings.

## What is Cognee?

**Cognee** is an open-source semantic memory layer that transforms unstructured, structured, semi-structured data into queryable knowledge graphs backed by embeddings. Cognee:

- **Automatically extracts** entities, relationships, and semantic meaning from text
- **Creates knowledge graphs** with embeddings 
- **Enables natural language queries** to retrieve relevant information
- **Maintains context** across different sessions and interactions
- **Supports multi-modal data** including text, documents, and structured data

## What is Google ADK? 

**Google Agent Development Kit (ADK)** is Google's framework for building AI agents. It provides:

- **Agent Abstraction**: Easy-to-use Agent class with instructions and tools
- **Tool Integration**: Seamless integration with external functions and APIs
- **Multiple Runners**: InMemoryRunner, VertexRunner for different deployment scenarios
- **Gemini Integration**: Native support for Google's Gemini models

## Why Combine Them?

The Cognee-Google ADK integration provides AI agents with persistent, semantic memory that survives between sessions. Agents can store information in Cognee's knowledge graph and retrieve it using natural language queries, enabling more accurate retrieval, continuity across conversations without manual state management.

Let's explore how this works in practice!


## 📋 Prerequisites

Before running this notebook, make sure you have the required dependencies installed:

```bash
# Using uv
uv sync

# Or using pip
pip install cognee-integration-google-adk
```

**Note**: You'll need both an OpenAI API key (for Cognee's LLM operations) and a Google API key (for Gemini models).


## Environment Setup

Both Google ADK and Cognee require API keys for LLM operations. Let's configure the environment:

In [2]:
import os

from dotenv import load_dotenv

os.environ["REQUIRE_AUTHENTICATION"] = "false"
os.environ["ENABLE_BACKEND_ACCESS_CONTROL"] = "false"


# Load API keys from .env file
# Get your OpenAI API key from: https://platform.openai.com/api-keys
# Get your Google API key from: https://aistudio.google.com/apikey
load_dotenv()

True

## Google ADK Fundamentals

Before diving into the Cognee integration, let's understand Google ADK's core building blocks:

### Agent
**Agent** is the central abstraction in Google ADK. It encapsulates:
- A model (e.g., `gemini-2.5-flash`)
- A name and description
- Instructions for behavior
- A list of tools it can use

### Runners
**Runners** execute agents and manage their lifecycle:
- **InMemoryRunner**: For local development and testing
- **VertexRunner**: For production deployment on Google Cloud

### Tools
**Tools** are functions that agents can call. Google ADK provides:
- **FunctionTool**: For simple synchronous functions
- **LongRunningFunctionTool**: For async operations (what we use with Cognee)

Let's create a simple agent to understand the basics:


In [5]:
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner

# Create a simple chatbot agent without tools
simple_agent = Agent(
    model="gemini-2.5-flash",
    name="simple_chatbot",
    description="A helpful assistant that answers questions",
    instruction="You are a helpful assistant. Answer questions concisely and accurately.",
    tools=[],  # No tools for now
)

print(f"Agent created: {simple_agent.name}")
print(f"Model: {simple_agent.model}")

Agent created: simple_chatbot
Model: gemini-2.5-flash


### Running the Agent

Now let's use the InMemoryRunner to execute our agent with a simple question:


In [6]:
# Create a runner for our agent
runner = InMemoryRunner(agent=simple_agent)

# Ask a simple question
events = await runner.run_debug("What is the capital of Argentina?")

# Print the agent's response
print("\n=== AGENT RESPONSE ===")
for event in events:
    if event.is_final_response() and event.content:
        for part in event.content.parts:
            if part.text:
                print(part.text)


Sending out request, model: gemini-2.5-flash, backend: GoogleLLMVariant.GEMINI_API, stream: False

AFC is enabled with max remote calls: 10.



 ### Created new session: debug_session_id

User > What is the capital of Argentina?



Response received from the model.


simple_chatbot > The capital of Argentina is Buenos Aires.

=== AGENT RESPONSE ===
The capital of Argentina is Buenos Aires.


## Adding Tools to Google ADK Agents

Our current agent can only chat - it has no access to external information or storage. Let's understand how tools work in Google ADK.

### Tool Types in Google ADK

Google ADK provides several tool types:

1. **FunctionTool**: Wraps a synchronous Python function
2. **LongRunningFunctionTool**: Wraps an async function (ideal for I/O operations)
3. **AgentTool**: Allows agents to call other agents

For Cognee integration, we use **LongRunningFunctionTool** because Cognee operations are asynchronous:

```python
from google.adk.tools import LongRunningFunctionTool

async def my_async_function(query: str) -> str:
    # Async operation here
    return result

my_tool = LongRunningFunctionTool(my_async_function)
```

Let's create a simple custom tool to demonstrate:


In [7]:
import asyncio

from google.adk.tools import LongRunningFunctionTool


# A simple async tool that simulates a database lookup
async def lookup_company_info(company_name: str) -> str:
    """
    Look up information about a company.

    Args:
        company_name: The name of the company to look up.

    Returns:
        Information about the company.
    """
    # Simulate async database lookup
    await asyncio.sleep(0.1)

    companies = {
        "acme": "ACME Corp - Founded 1920, specializes in cartoon physics products.",
        "cognee": "Cognee - AI semantic memory layer for intelligent agents.",
    }

    company_key = company_name.lower()
    return companies.get(company_key, f"No information found for {company_name}")


# Wrap the async function as a LongRunningFunctionTool
company_lookup_tool = LongRunningFunctionTool(lookup_company_info)

print(f"Tool created: {company_lookup_tool.func.__name__}")

Tool created: lookup_company_info


### Using Tools with an Agent

Now let's create an agent that uses our custom tool:


In [8]:
# Create an agent with our tool
agent_with_tool = Agent(
    model="gemini-2.5-flash",
    name="company_researcher",
    description="An assistant that can look up company information",
    instruction="You are a company researcher. Use the lookup_company_info tool to find information about companies when asked.",
    tools=[company_lookup_tool],
)

runner = InMemoryRunner(agent=agent_with_tool)

# Ask about a company
events = await runner.run_debug("What can you tell me about Cognee?")

print("\n=== AGENT RESPONSE ===")
for event in events:
    if event.is_final_response() and event.content:
        for part in event.content.parts:
            if part.text:
                print(part.text)


Sending out request, model: gemini-2.5-flash, backend: GoogleLLMVariant.GEMINI_API, stream: False



 ### Created new session: debug_session_id

User > What can you tell me about Cognee?



Response received from the model.


Sending out request, model: gemini-2.5-flash, backend: GoogleLLMVariant.GEMINI_API, stream: False

Response received from the model.


company_researcher > Cognee is an AI semantic memory layer designed for intelligent agents.

=== AGENT RESPONSE ===
Cognee is an AI semantic memory layer designed for intelligent agents.


## Introducing Cognee to Google ADK: Semantic Memory for Agents

Now let's add **persistent semantic memory** to our agents using Cognee!

### What Cognee Adds to Google ADK:

- **Semantic Memory**: Store and retrieve information using natural language from Cognee's knowledge graph backed by embeddings
- **Knowledge Graphs**: Automatic entity and relationship extraction
- **Cross-Session Persistence**: Memory survives between different agent instances
- **Intelligent Search**: Find relevant information by meaning using Cognee's advanced retrieval methods
- **Session Isolation**: Keep different users' data separate and secure

### Cognee Tools Integration

The integration provides two main tools:
- **`add_tool`**: Store information in Cognee's knowledge graph
- **`search_tool`**: Query stored information using natural language

Let's start fresh by pruning any existing Cognee data:


In [9]:
import cognee

# Clean slate - remove any existing data
await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)

print("✓ Cognee data pruned successfully!")


2026-03-09T15:59:04.629675 [info     ] Deleted Kuzu database files at /Users/handekafkas/Documents/local-code/cognee-integrations/integrations/google-adk/.venv/lib/python3.13/site-packages/cognee/.cognee_system/databases/cognee_graph_kuzu [cognee.shared.logging_utils]

2026-03-09T15:59:06.636083 [info     ] Database deleted successfully. [cognee.shared.logging_utils]

Deleting cache...             

✓ Cache deleted successfully! 


✓ Cognee data pruned successfully!


### Importing Cognee Tools

Now let's import the Cognee tools and create an agent with semantic memory capabilities:


In [10]:
from cognee_integration_google_adk import add_tool, search_tool

# Create an agent with Cognee memory tools
memory_agent = Agent(
    model="gemini-2.5-flash",
    name="memory_agent",
    description="An assistant with persistent memory capabilities",
    instruction="""You are a helpful assistant with access to a persistent knowledge base.
    
    When the user gives you information to remember, use the add_tool to store it.
    When the user asks questions about previously stored information, use the search_tool to find it.
    
    Always confirm when you've stored information and provide relevant results when searching.""",
    tools=[add_tool, search_tool],
)

print(f"✓ Memory agent created with tools: {[t.func.__name__ for t in memory_agent.tools]}")

✓ Memory agent created with tools: ['add_tool_impl', 'search_tool_impl']


### Storing Information with Cognee

Let's give our agent some contract information to remember. The agent will use the `add_tool` to store this in Cognee's knowledge graph:


In [11]:
runner = InMemoryRunner(agent=memory_agent)

# Add contract information
contracts = [
    'We have signed a contract with "Meditech Solutions". Company is in the healthcare industry. Start date: Jan 2023, End date: Dec 2025. Contract value: £1.2M.',
    'We have signed a contract with "QuantumSoft". Company is in the technology industry. Start date: Aug 2024, End date: Aug 2028. Contract value: £5.5M.',
    'We have signed a contract with "Orion Retail Group". Company is in the retail industry. Start date: Mar 2024, End date: Mar 2026. Contract value: £850K.',
]

print("=== ADDING CONTRACTS TO MEMORY ===")
for i, contract in enumerate(contracts, 1):
    print(f"\n[{i}/3] Adding: {contract[:60]}...")
    events = await runner.run_debug(f"Please remember this information: {contract}")

    # Show confirmation
    for event in events:
        if event.is_final_response() and event.content:
            for part in event.content.parts:
                if part.text:
                    print(
                        f"   Agent: {part.text[:100]}..."
                        if len(part.text) > 100
                        else f"   Agent: {part.text}"
                    )

print("\n✓ All contracts added to Cognee memory!")


Sending out request, model: gemini-2.5-flash, backend: GoogleLLMVariant.GEMINI_API, stream: False


=== ADDING CONTRACTS TO MEMORY ===

[1/3] Adding: We have signed a contract with "Meditech Solutions". Company...

 ### Created new session: debug_session_id

User > Please remember this information: We have signed a contract with "Meditech Solutions". Company is in the healthcare industry. Start date: Jan 2023, End date: Dec 2025. Contract value: £1.2M.



Response received from the model.

Adding data to cognee: Contract with "Meditech Solutions". Healthcare industry. Start date: Jan 2023, End date: Dec 2025. Contract value: £1.2M.

2026-03-09T16:00:43.105471 [info     ] Testing connection to LLM endpoint... [cognee.shared.logging_utils]


User f40b471e-dc9c-46e6-8a12-6cfbd2874e5f has registered.



2026-03-09T16:00:47.024429 [info     ] Testing connection to Embedding endpoint... [cognee.shared.logging_utils]

2026-03-09T16:00:50.513204 [info     ] Pipeline run started: `42b20a88-adc5-5830-b8c1-6f52dd164787` [run_tasks_with_telemetry()]

2026-03-09T16:00:50.776839 [info     ] Coroutine task started: `resolve_data_directories` [run_tasks_base]

2026-03-09T16:00:50.997750 [info     ] Coroutine task started: `ingest_data` [run_tasks_base]

2026-03-09T16:00:51.203986 [info     ] Registered loader: pypdf_loader [cognee.infrastructure.loaders.LoaderEngine]

2026-03-09T16:00:51.205414 [info     ] Registered loader: text_loader [cognee.infrastructure.loaders.LoaderEngine]

2026-03-09T16:00:51.205945 [info     ] Registered loader: image_loader [cognee.infrastructure.loaders.LoaderEngine]

2026-03-09T16:00:51.206630 [info     ] Registered loader: audio_loader [cognee.infrastructure.loaders.LoaderEngine]

2026-03-09T16:00:51.207088 [info     ] Registered loader: csv_loader  [cognee.infrast

memory_agent > I have remembered the following information: Contract with "Meditech Solutions". Healthcare industry. Start date: Jan 2023, End date: Dec 2025. Contract value: £1.2M.
   Agent: I have remembered the following information: Contract with "Meditech Solutions". Healthcare industry...

[2/3] Adding: We have signed a contract with "QuantumSoft". Company is in ...

 ### Continue session: debug_session_id

User > Please remember this information: We have signed a contract with "QuantumSoft". Company is in the technology industry. Start date: Aug 2024, End date: Aug 2028. Contract value: £5.5M.



Response received from the model.

Adding data to cognee: Contract with "QuantumSoft". Technology industry. Start date: Aug 2024, End date: Aug 2028. Contract value: £5.5M.

2026-03-09T16:05:49.784059 [info     ] Pipeline run started: `42b20a88-adc5-5830-b8c1-6f52dd164787` [run_tasks_with_telemetry()]

2026-03-09T16:05:49.980798 [info     ] Coroutine task started: `resolve_data_directories` [run_tasks_base]

2026-03-09T16:05:50.191518 [info     ] Coroutine task started: `ingest_data` [run_tasks_base]

2026-03-09T16:05:50.667086 [info     ] Coroutine task completed: `ingest_data` [run_tasks_base]

2026-03-09T16:05:50.871553 [info     ] Coroutine task completed: `resolve_data_directories` [run_tasks_base]

2026-03-09T16:05:51.057644 [info     ] Pipeline run completed: `42b20a88-adc5-5830-b8c1-6f52dd164787` [run_tasks_with_telemetry()]

2026-03-09T16:05:53.310162 [info     ] No ontology file provided. No owl ontology will be attached to the graph. [OntologyAdapter]

2026-03-09T16:05:53.3

memory_agent > I have remembered the following information: Contract with "QuantumSoft". Technology industry. Start date: Aug 2024, End date: Aug 2028. Contract value: £5.5M.
   Agent: I have remembered the following information: Contract with "QuantumSoft". Technology industry. Start...

[3/3] Adding: We have signed a contract with "Orion Retail Group". Company...

 ### Continue session: debug_session_id

User > Please remember this information: We have signed a contract with "Orion Retail Group". Company is in the retail industry. Start date: Mar 2024, End date: Mar 2026. Contract value: £850K.



Response received from the model.

Adding data to cognee: Contract with "Orion Retail Group". Retail industry. Start date: Mar 2024, End date: Mar 2026. Contract value: £850K.

2026-03-09T16:07:01.205274 [info     ] Pipeline run started: `42b20a88-adc5-5830-b8c1-6f52dd164787` [run_tasks_with_telemetry()]

2026-03-09T16:07:01.387968 [info     ] Coroutine task started: `resolve_data_directories` [run_tasks_base]

2026-03-09T16:07:01.583273 [info     ] Coroutine task started: `ingest_data` [run_tasks_base]

2026-03-09T16:07:01.824236 [info     ] Coroutine task completed: `ingest_data` [run_tasks_base]

2026-03-09T16:07:02.064039 [info     ] Coroutine task completed: `resolve_data_directories` [run_tasks_base]

2026-03-09T16:07:02.269544 [info     ] Pipeline run completed: `42b20a88-adc5-5830-b8c1-6f52dd164787` [run_tasks_with_telemetry()]

2026-03-09T16:07:04.498742 [info     ] No ontology file provided. No owl ontology will be attached to the graph. [OntologyAdapter]

2026-03-09T16:07:0

memory_agent > I have remembered the following information: Contract with "Orion Retail Group". Retail industry. Start date: Mar 2024, End date: Mar 2026. Contract value: £850K.
   Agent: I have remembered the following information: Contract with "Orion Retail Group". Retail industry. St...

✓ All contracts added to Cognee memory!


### Background Data Ingestion

Let's also add some additional data directly to Cognee (bypassing the agent) to demonstrate how the agent can access pre-existing knowledge. This simulates a scenario where:
- **Historical data** exists in the knowledge base
- **Agent interactions** add new information
- **Both sources** are searchable together


In [12]:
# Add data directly to Cognee (simulating pre-existing knowledge base)
additional_contracts = """
We have signed a contract with the following company: "HealthBridge Systems". Company is in the healthcare industry. Start date is Feb 2023 and end date is Jan 2026. Contract value is £2.4M.
We have signed a contract with the following company: "NovaCare Diagnostics". Company is in the healthcare industry. Start date is May 2024 and end date is Apr 2027. Contract value is £1.6M.
We have signed a contract with the following company: "SkyPort Logistics". Company is in the logistics industry. Start date is Nov 2022 and end date is Oct 2026. Contract value is £3.1M.
"""

print("Adding background data directly to Cognee...")
await cognee.add(additional_contracts)
await cognee.cognify()
print("✓ Background data processed and added to knowledge graph!")


2026-03-09T16:08:18.931091 [info     ] Pipeline run started: `42b20a88-adc5-5830-b8c1-6f52dd164787` [run_tasks_with_telemetry()]


Adding background data directly to Cognee...



2026-03-09T16:08:19.176172 [info     ] Coroutine task started: `resolve_data_directories` [run_tasks_base]

2026-03-09T16:08:19.476081 [info     ] Coroutine task started: `ingest_data` [run_tasks_base]

2026-03-09T16:08:19.752409 [info     ] Coroutine task completed: `ingest_data` [run_tasks_base]

2026-03-09T16:08:19.993087 [info     ] Coroutine task completed: `resolve_data_directories` [run_tasks_base]

2026-03-09T16:08:20.220095 [info     ] Pipeline run completed: `42b20a88-adc5-5830-b8c1-6f52dd164787` [run_tasks_with_telemetry()]

2026-03-09T16:08:20.431144 [info     ] No ontology file provided. No owl ontology will be attached to the graph. [OntologyAdapter]

2026-03-09T16:08:20.473016 [info     ] Pipeline run started: `a9442bfe-a11c-536d-92f1-df1ec55567bb` [run_tasks_with_telemetry()]

2026-03-09T16:08:20.679342 [info     ] Coroutine task started: `classify_documents` [run_tasks_base]

2026-03-09T16:08:20.919853 [info     ] Async Generator task started: `extract_chunks_from_doc

✓ Background data processed and added to knowledge graph!


### Visualizing the Knowledge Graph

Let's visualize what Cognee has built from our data:


In [13]:
import webbrowser


async def visualize_graph(file_name, open_browser=True):
    destination_file_path = os.path.join(os.getcwd(), file_name)
    await cognee.visualize_graph(destination_file_path)

    if open_browser:
        url = "file://" + os.path.abspath(destination_file_path)
        webbrowser.open(url)

    print(f"✓ Graph visualization saved to: {destination_file_path}")


await visualize_graph(file_name="first_graph_visualization.html")


2026-03-09T16:12:34.159949 [info     ] Retrieved 54 nodes and 114 edges in 0.01 seconds [cognee.shared.logging_utils]

2026-03-09T16:12:34.163691 [info     ] Graph visualization saved as /Users/handekafkas/Documents/local-code/cognee-integrations/integrations/google-adk/cognee_integration_google_adk/first_graph_visualization.html [cognee.shared.logging_utils]

2026-03-09T16:12:34.164219 [info     ] The HTML file has been stored at path: /Users/handekafkas/Documents/local-code/cognee-integrations/integrations/google-adk/cognee_integration_google_adk/first_graph_visualization.html [cognee.shared.logging_utils]


✓ Graph visualization saved to: /Users/handekafkas/Documents/local-code/cognee-integrations/integrations/google-adk/cognee_integration_google_adk/first_graph_visualization.html


Open the generated HTML file in your browser to explore the knowledge graph. You'll see:
- Entities extracted from the contracts (companies, industries, values)
- Relationships between entities
- Data from both agent interactions and background ingestion


## Cross-Session Memory Persistence

Now let's demonstrate one of Cognee's key features: **persistent memory across agent instances**.

We'll create a completely fresh agent instance that:
- Has **no conversation history** from the previous agent
- Has **no internal state** carried over
- **CAN access** all information stored in Cognee's knowledge graph

This shows how Cognee provides **true persistent memory** that survives agent restarts:


In [14]:
# Create a FRESH agent instance - no shared state with previous agent
fresh_agent = Agent(
    model="gemini-2.5-flash",
    name="fresh_research_agent",
    description="A research analyst with access to the company knowledge base",
    instruction="""You are an expert research analyst with access to a comprehensive knowledge base about company contracts.
    Use the search_tool to find information about contracts when asked.
    Provide detailed, structured responses with all relevant contract details.""",
    tools=[add_tool, search_tool],
)

fresh_runner = InMemoryRunner(agent=fresh_agent)

# Search for healthcare contracts - the fresh agent has no memory of what we added,
# but can still find it through Cognee!
search_query = "I need to research our contract portfolio. Can you search for any contracts we have with companies in the healthcare industry?"

print("=== SEARCHING WITH FRESH AGENT ===")
print(f"Query: {search_query}\n")

events = await fresh_runner.run_debug(search_query)

print("=== AGENT RESPONSE ===")
for event in events:
    if event.is_final_response() and event.content:
        for part in event.content.parts:
            if part.text:
                print(part.text)


Sending out request, model: gemini-2.5-flash, backend: GoogleLLMVariant.GEMINI_API, stream: False


=== SEARCHING WITH FRESH AGENT ===
Query: I need to research our contract portfolio. Can you search for any contracts we have with companies in the healthcare industry?


 ### Created new session: debug_session_id

User > I need to research our contract portfolio. Can you search for any contracts we have with companies in the healthcare industry?



Response received from the model.

Searching cognee for: contracts with companies in the healthcare industry with node_set: None

2026-03-09T16:12:59.324310 [info     ] Vector collection retrieval completed: Retrieved distances from 6 collections in 3.65s [cognee.shared.logging_utils]

2026-03-09T16:12:59.325041 [info     ] Retrieving ID-filtered graph from database. [CogneeGraph]

2026-03-09T16:12:59.335249 [info     ] ID-filtered retrieval: 54 nodes and 114 edges in 0.01s [cognee.shared.logging_utils]

2026-03-09T16:12:59.340405 [info     ] Graph projection completed: 54 nodes, 114 edges in 0.00s [CogneeGraph]

2026-03-09T16:12:59.345446 [info     ] Completed resolving edges to text [cognee.shared.logging_utils] extra={'node_count': 51, 'connection_count': 100}

Search results: ['HealthBridge Systems — Feb 2023 to Jan 2026 — £2,400,000\nNovaCare Diagnostics — May 2024 to Apr 2027 — £1,600,000\nMeditech Solutions — Jan 2023 to Dec 2025 — £1,200,000']

Sending out request, model: gemi

fresh_research_agent > Here are the contracts found with companies in the healthcare industry:

*   **HealthBridge Systems:**
    *   **Term:** February 2023 to January 2026
    *   **Value:** £2,400,000

*   **NovaCare Diagnostics:**
    *   **Term:** May 2024 to April 2027
    *   **Value:** £1,600,000

*   **Meditech Solutions:**
    *   **Term:** January 2023 to December 2025
    *   **Value:** £1,200,000
=== AGENT RESPONSE ===
Here are the contracts found with companies in the healthcare industry:

*   **HealthBridge Systems:**
    *   **Term:** February 2023 to January 2026
    *   **Value:** £2,400,000

*   **NovaCare Diagnostics:**
    *   **Term:** May 2024 to April 2027
    *   **Value:** £1,600,000

*   **Meditech Solutions:**
    *   **Term:** January 2023 to December 2025
    *   **Value:** £1,200,000


**Amazing!** The fresh agent found all the healthcare contracts, including:
- Contracts added through agent interactions (Meditech Solutions)
- Contracts added through background ingestion (HealthBridge Systems, NovaCare Diagnostics)

This demonstrates true **cross-session persistence** - the knowledge survives agent restarts!


## Sessions with Cognee

In multi-user applications, you often need to **isolate data between users** while still maintaining persistent memory for each user. Cognee supports this through **session-based tools**.

### How Session Isolation Works

The `get_sessionized_cognee_tools()` function creates tools that are bound to a specific session ID:

```python
def get_sessionized_cognee_tools(session_id: Optional[str] = None) -> list:
    """
    Returns a list of cognee tools sessionized for a specific user.
    
    Args:
        session_id: The session ID to bind to all tools.
                    If None, a UUID-based ID is auto-generated.
    
    Returns:
        list: [sessionized_add_tool, sessionized_search_tool]
    """
```

When you use sessionized tools:
- Data added is tagged with the session ID
- Searches only return data from that session
- Different sessions are completely isolated


In [15]:
# Start fresh for the session demo
await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✓ Starting fresh for session demonstration")


2026-03-09T16:13:24.205512 [info     ] Deleted Kuzu database files at /Users/handekafkas/Documents/local-code/cognee-integrations/integrations/google-adk/.venv/lib/python3.13/site-packages/cognee/.cognee_system/databases/cognee_graph_kuzu [cognee.shared.logging_utils]

2026-03-09T16:13:26.258604 [info     ] Database deleted successfully. [cognee.shared.logging_utils]

Deleting cache...             

✓ Cache deleted successfully! 


✓ Starting fresh for session demonstration


### Creating Session-Isolated Agents

Let's create two agents for two different users, each with their own isolated memory:


In [16]:
from cognee_integration_google_adk import get_sessionized_cognee_tools

# Create sessionized tools for User A (Alice)
alice_add_tool, alice_search_tool = get_sessionized_cognee_tools("alice-session")

alice_agent = Agent(
    model="gemini-2.5-flash",
    name="alice_assistant",
    description="Alice's personal assistant with isolated memory",
    instruction="You are Alice's personal assistant. Store and retrieve information for Alice.",
    tools=[alice_add_tool, alice_search_tool],
)

# Create sessionized tools for User B (Bob)
bob_add_tool, bob_search_tool = get_sessionized_cognee_tools("bob-session")

bob_agent = Agent(
    model="gemini-2.5-flash",
    name="bob_assistant",
    description="Bob's personal assistant with isolated memory",
    instruction="You are Bob's personal assistant. Store and retrieve information for Bob.",
    tools=[bob_add_tool, bob_search_tool],
)

print("✓ Created isolated agents for Alice and Bob")


Initialized session with session_id = alice-session

Initialized session with session_id = bob-session


✓ Created isolated agents for Alice and Bob


### Adding Data to Different Sessions

Let's add different contracts to each user's session:


In [17]:
# Alice's contracts (Insurance industry)
alice_runner = InMemoryRunner(agent=alice_agent)

alice_contracts = [
    'Contract with "Guardian Insurance Ltd" - Insurance industry, Feb 2023 to Feb 2026, £1.8M',
    'Contract with "Pioneer Assurance Group" - Insurance industry, Oct 2024 to Oct 2029, £4.2M',
]

print("=== ALICE'S SESSION ===")
for contract in alice_contracts:
    print(f"Adding: {contract[:50]}...")
    await alice_runner.run_debug(f"Remember this: {contract}")

# Bob's contracts (Technology industry)
bob_runner = InMemoryRunner(agent=bob_agent)

bob_contracts = [
    'Contract with "TechNova Solutions" - Technology industry, Jan 2024 to Jan 2027, £3.5M',
    'Contract with "DataStream Analytics" - Technology industry, Mar 2024 to Mar 2028, £2.1M',
]

print("\n=== BOB'S SESSION ===")
for contract in bob_contracts:
    print(f"Adding: {contract[:50]}...")
    await bob_runner.run_debug(f"Remember this: {contract}")

print("\n✓ Contracts added to separate sessions!")


Sending out request, model: gemini-2.5-flash, backend: GoogleLLMVariant.GEMINI_API, stream: False


=== ALICE'S SESSION ===
Adding: Contract with "Guardian Insurance Ltd" - Insurance...

 ### Created new session: debug_session_id

User > Remember this: Contract with "Guardian Insurance Ltd" - Insurance industry, Feb 2023 to Feb 2026, £1.8M



Response received from the model.

Using tool add_tool_impl with user_id: alice-session

Adding data to cognee: Contract with "Guardian Insurance Ltd" - Insurance industry, Feb 2023 to Feb 2026, £1.8M

2026-03-09T16:13:40.101963 [info     ] Pipeline run started: `161f2e23-ec52-5720-b6d0-b8f44eb1bdcd` [run_tasks_with_telemetry()]


User 6c9ef9ad-ea2b-4909-97fc-25c649037071 has registered.



2026-03-09T16:13:40.481380 [info     ] Coroutine task started: `resolve_data_directories` [run_tasks_base]

2026-03-09T16:13:40.742271 [info     ] Coroutine task started: `ingest_data` [run_tasks_base]

2026-03-09T16:13:41.056704 [info     ] Coroutine task completed: `ingest_data` [run_tasks_base]

2026-03-09T16:13:41.362778 [info     ] Coroutine task completed: `resolve_data_directories` [run_tasks_base]

2026-03-09T16:13:41.592168 [info     ] Pipeline run completed: `161f2e23-ec52-5720-b6d0-b8f44eb1bdcd` [run_tasks_with_telemetry()]

2026-03-09T16:13:43.839184 [info     ] No ontology file provided. No owl ontology will be attached to the graph. [OntologyAdapter]

2026-03-09T16:13:43.936387 [info     ] Pipeline run started: `73a15584-116f-51d7-b68a-298963abb320` [run_tasks_with_telemetry()]

2026-03-09T16:13:44.241070 [info     ] Coroutine task started: `classify_documents` [run_tasks_base]

2026-03-09T16:13:44.543679 [info     ] Async Generator task started: `extract_chunks_from_doc

...

alice_assistant > I have remembered the contract information with "Pioneer Assurance Group".

=== BOB'S SESSION ===
Adding: Contract with "TechNova Solutions" - Technology in...

 ### Created new session: debug_session_id

User > Remember this: Contract with "TechNova Solutions" - Technology industry, Jan 2024 to Jan 2027, £3.5M



Response received from the model.

Using tool add_tool_impl with user_id: bob-session

Adding data to cognee: Contract with "TechNova Solutions" - Technology industry, Jan 2024 to Jan 2027, £3.5M

2026-03-09T16:16:23.974735 [info     ] Pipeline run started: `161f2e23-ec52-5720-b6d0-b8f44eb1bdcd` [run_tasks_with_telemetry()]

2026-03-09T16:16:24.233125 [info     ] Coroutine task started: `resolve_data_directories` [run_tasks_base]

2026-03-09T16:16:24.532378 [info     ] Coroutine task started: `ingest_data` [run_tasks_base]

2026-03-09T16:16:24.888940 [info     ] Coroutine task completed: `ingest_data` [run_tasks_base]

2026-03-09T16:16:25.252949 [info     ] Coroutine task completed: `resolve_data_directories` [run_tasks_base]

2026-03-09T16:16:25.575517 [info     ] Pipeline run completed: `161f2e23-ec52-5720-b6d0-b8f44eb1bdcd` [run_tasks_with_telemetry()]

2026-03-09T16:16:27.965455 [info     ] No ontology file provided. No owl ontology will be attached to the graph. [OntologyAdapter]

bob_assistant > I've stored the contract details for "TechNova Solutions" which runs from Jan 2024 to Jan 2027 for £3.5M.
Adding: Contract with "DataStream Analytics" - Technology ...

 ### Continue session: debug_session_id

User > Remember this: Contract with "DataStream Analytics" - Technology industry, Mar 2024 to Mar 2028, £2.1M



Response received from the model.

Using tool add_tool_impl with user_id: bob-session

Adding data to cognee: Contract with "DataStream Analytics" - Technology industry, Mar 2024 to Mar 2028, £2.1M

2026-03-09T16:17:30.150674 [info     ] Pipeline run started: `161f2e23-ec52-5720-b6d0-b8f44eb1bdcd` [run_tasks_with_telemetry()]

2026-03-09T16:17:30.480234 [info     ] Coroutine task started: `resolve_data_directories` [run_tasks_base]

2026-03-09T16:17:30.730058 [info     ] Coroutine task started: `ingest_data` [run_tasks_base]

2026-03-09T16:17:31.157624 [info     ] Coroutine task completed: `ingest_data` [run_tasks_base]

2026-03-09T16:17:31.419958 [info     ] Coroutine task completed: `resolve_data_directories` [run_tasks_base]

2026-03-09T16:17:31.745414 [info     ] Pipeline run completed: `161f2e23-ec52-5720-b6d0-b8f44eb1bdcd` [run_tasks_with_telemetry()]

2026-03-09T16:17:34.023106 [info     ] No ontology file provided. No owl ontology will be attached to the graph. [OntologyAdapte

bob_assistant > I've stored the contract details for "DataStream Analytics" which runs from Mar 2024 to Mar 2028 for £2.1M.

✓ Contracts added to separate sessions!


### Demonstrating Session Isolation

Now let's verify that each user can only see their own data:


In [18]:
# Alice searches for contracts
print("=== ALICE SEARCHING ===")
alice_events = await alice_runner.run_debug("What contracts do we have?")

for event in alice_events:
    if event.is_final_response() and event.content:
        for part in event.content.parts:
            if part.text:
                print(f"Alice's results:\n{part.text}")

print("\n" + "=" * 50 + "\n")

# Bob searches for contracts
print("=== BOB SEARCHING ===")
bob_events = await bob_runner.run_debug("What contracts do we have?")

for event in bob_events:
    if event.is_final_response() and event.content:
        for part in event.content.parts:
            if part.text:
                print(f"Bob's results:\n{part.text}")


Sending out request, model: gemini-2.5-flash, backend: GoogleLLMVariant.GEMINI_API, stream: False


=== ALICE SEARCHING ===

 ### Continue session: debug_session_id

User > What contracts do we have?



Response received from the model.

Using tool search_tool_impl with user_id: alice-session

Searching cognee for: What contracts do we have? with node_set: ['alice-session']

2026-03-09T16:23:21.159677 [info     ] Vector collection retrieval completed: Retrieved distances from 3 collections in 0.57s [cognee.shared.logging_utils]

2026-03-09T16:23:21.160395 [info     ] Retrieving graph filtered by node type and node name (NodeSet). [CogneeGraph]

2026-03-09T16:23:21.180960 [info     ] Graph projection completed: 17 nodes, 84 edges in 0.01s [CogneeGraph]

2026-03-09T16:23:21.184687 [info     ] Completed resolving edges to text [cognee.shared.logging_utils] extra={'node_count': 17, 'connection_count': 84}

Search results: ['Two contracts: 1) Guardian Insurance Ltd — Feb 2023 to Feb 2026 — £1.8M. 2) Pioneer Assurance Group — Oct 2024 to Oct 2029 — £4.2M.']

Sending out request, model: gemini-2.5-flash, backend: GoogleLLMVariant.GEMINI_API, stream: False

Response received from the model.


alice_assistant > We have two contracts:
1) Guardian Insurance Ltd - February 2023 to February 2026, for £1.8M.
2) Pioneer Assurance Group - October 2024 to October 2029, for £4.2M.
Alice's results:
We have two contracts:
1) Guardian Insurance Ltd - February 2023 to February 2026, for £1.8M.
2) Pioneer Assurance Group - October 2024 to October 2029, for £4.2M.


=== BOB SEARCHING ===

 ### Continue session: debug_session_id

User > What contracts do we have?



Response received from the model.

Using tool search_tool_impl with user_id: bob-session

Searching cognee for: contracts with node_set: ['bob-session']

2026-03-09T16:23:29.086098 [info     ] Vector collection retrieval completed: Retrieved distances from 3 collections in 0.36s [cognee.shared.logging_utils]

2026-03-09T16:23:29.086629 [info     ] Retrieving graph filtered by node type and node name (NodeSet). [CogneeGraph]

2026-03-09T16:23:29.096104 [info     ] Graph projection completed: 17 nodes, 84 edges in 0.00s [CogneeGraph]

2026-03-09T16:23:29.102079 [info     ] Completed resolving edges to text [cognee.shared.logging_utils] extra={'node_count': 17, 'connection_count': 84}

Search results: ['Contracts:\n- Contract with DataStream Analytics — Technology; Mar 2024 to Mar 2028; £2.1M\n- Contract with TechNova Solutions — Technology; Jan 2024 to Jan 2027; £3.5M']

Sending out request, model: gemini-2.5-flash, backend: GoogleLLMVariant.GEMINI_API, stream: False

Response received 

bob_assistant > We have a contract with "DataStream Analytics" in the Technology industry from March 2024 to March 2028 for £2.1M, and a contract with "TechNova Solutions" also in the Technology industry from January 2024 to January 2027 for £3.5M.
Bob's results:
We have a contract with "DataStream Analytics" in the Technology industry from March 2024 to March 2028 for £2.1M, and a contract with "TechNova Solutions" also in the Technology industry from January 2024 to January 2027 for £3.5M.


**Notice:** 
- Alice only sees her insurance contracts
- Bob only sees his technology contracts
- Complete session isolation is maintained!


### Visualizing Session Clusters

Let's visualize the knowledge graph to see how sessions create distinct clusters:


In [19]:
await visualize_graph(file_name="sessionized_graph_visualization.html")


2026-03-09T16:25:08.661329 [info     ] Retrieved 45 nodes and 112 edges in 0.03 seconds [cognee.shared.logging_utils]

2026-03-09T16:25:08.671358 [info     ] Graph visualization saved as /Users/handekafkas/Documents/local-code/cognee-integrations/integrations/google-adk/cognee_integration_google_adk/sessionized_graph_visualization.html [cognee.shared.logging_utils]

2026-03-09T16:25:08.672286 [info     ] The HTML file has been stored at path: /Users/handekafkas/Documents/local-code/cognee-integrations/integrations/google-adk/cognee_integration_google_adk/sessionized_graph_visualization.html [cognee.shared.logging_utils]


✓ Graph visualization saved to: /Users/handekafkas/Documents/local-code/cognee-integrations/integrations/google-adk/cognee_integration_google_adk/sessionized_graph_visualization.html


In the visualization, you'll observe **session-based data clustering**:

1. **Alice's Cluster**: Data from `alice-session` grouped together
2. **Bob's Cluster**: Data from `bob-session` in a separate cluster
3. **Global Data**: Information added outside sessions forms its own cluster

This demonstrates how Cognee maintains both **session isolation** and **global knowledge accessibility**.


## Understanding Session ID Generation

Let's explore how session management works under the hood.

### Automatic Session IDs

If you don't provide a session ID, one is auto-generated:

```python
# Auto-generated session ID
add_tool, search_tool = get_sessionized_cognee_tools()
# Results in: cognee-test-user-<uuid>
```

### Custom Session IDs

For production applications, you'll want to use meaningful session IDs based on:
- User authentication tokens
- Organization IDs
- Custom identifiers from your system


In [20]:
# Example: Using authentication-based session IDs
async def get_user_session_id() -> str:
    """Simulate getting user ID from authentication system"""
    # In production, this would come from your auth system
    return "user-12345-authenticated"


# Create tools with auth-based session ID
user_session_id = await get_user_session_id()
auth_add_tool, auth_search_tool = get_sessionized_cognee_tools(user_session_id)

print(f"✓ Created tools with custom session ID: {user_session_id}")


Initialized session with session_id = user-12345-authenticated


✓ Created tools with custom session ID: user-12345-authenticated


## Summary

In this guide, we've learned how to:

### Google ADK Fundamentals
- ✅ Create agents with the `Agent` class
- ✅ Execute agents using `InMemoryRunner`
- ✅ Add tools using `LongRunningFunctionTool`

### Cognee Integration
- ✅ Use `add_tool` to store information in the knowledge graph
- ✅ Use `search_tool` to query stored information
- ✅ Demonstrate cross-session memory persistence
- ✅ Visualize knowledge graphs

### Session Management
- ✅ Create session-isolated tools with `get_sessionized_cognee_tools()`
- ✅ Maintain data isolation between users
- ✅ Use custom session IDs for production applications

### Key Takeaways

1. **Cognee provides persistent memory** that survives agent restarts
2. **Session isolation** keeps user data separate and secure
3. **Knowledge graphs** enable semantic search across stored information
4. **Easy integration** with Google ADK through simple tool imports

## Next Steps

- Explore advanced Cognee features like custom pipelines
- Integrate with other data sources (documents, PDFs)
- Deploy to production using VertexRunner
- Check out the [Cognee documentation](https://docs.cognee.ai) for more capabilities
